In [1]:
def fetch_image_data():
    # TODO
    return

In [2]:
from pathlib import Path
import os

IMAGES_PATH = Path("../resources/cloud-images/NASA_GLOBE_CD/downloaded_images")

def index_labeled_images(images_path=IMAGES_PATH):
    cloud_labels = []
    images_path = Path(images_path)
    labeled_images = {}
    if not images_path.exists():
        return labeled_images

    for cloud_dir in sorted(p for p in images_path.iterdir() if p.is_dir()):
        count = 0
        for img_path in sorted(p for p in cloud_dir.iterdir() if p.is_file()):
            if count > 3000:
                break
            labeled_images[img_path.name] = {
                "label": cloud_dir.name,
                "path": str(img_path)
            }
            count += 1

    return labeled_images, cloud_labels

In [3]:
labeled_images, cloud_labels = index_labeled_images()

In [4]:
import os
import matplotlib.pyplot as plt
import numpy as np
import cv2
from tqdm.auto import tqdm

def extract_labels(labeled_images):
    import os  # Ensure os is imported for path validation
    images = []
    labels = []
    for (i, image_name) in tqdm(enumerate(labeled_images), total=len(labeled_images)):
        path = labeled_images[image_name]['path']

        # Check if the file exists before attempting to read
        if not os.path.exists(path):
            print(f"Error: File does not exist at path: {path}")
            continue

        # Attempt to read the image
        image = cv2.imread(path)
        if image is None:
            print(f"Error: Unable to read image at path: {path}")
            continue

        try:
            # Convert to grayscale and resize
            image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
            image = cv2.resize(image, (64, 64))
            images.append(image)

            # Append the label
            label = labeled_images[image_name]['label']
            labels.append(label)
        except Exception as e:
            print(f"Error processing image {image_name}: {e}")
            continue

    return np.array(images), np.array(labels)

/Users/lucasnseyep/code/LucasNseyep/belle-weder/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
images, labels = extract_labels(labeled_images)

100%|██████████| 30559/30559 [04:13<00:00, 120.46it/s]


In [6]:
print(images)

[[[133 136 137 ... 154 152 149]
  [131 136 138 ... 154 152 149]
  [133 135 138 ... 154 150 148]
  ...
  [ 14  15  22 ...  16  20  16]
  [ 14  27  19 ...  17  19  18]
  [ 18  21  18 ...  16  18  14]]

 [[122 123 123 ... 130 131 129]
  [122 123 123 ... 132 129 130]
  [122 124 121 ... 132 132 131]
  ...
  [162 193 202 ... 205 204 177]
  [ 93  97 115 ... 125 156 157]
  [ 74 104 134 ... 127 180 103]]

 [[ 99  99 100 ... 193 193 193]
  [ 98 101 100 ... 194 194 194]
  [101 102 102 ... 194 193 193]
  ...
  [160 160  18 ...  24  17  17]
  [148  47  65 ...  14  14  14]
  [ 22  33  27 ...  15  14  26]]

 ...

 [[254 254 254 ... 254 254 254]
  [254 254 254 ... 254 254 254]
  [254 254 254 ... 254 254 254]
  ...
  [110 114 103 ... 108  93  94]
  [ 82  98 102 ...  89  99  93]
  [ 92 112 129 ...  93  90  88]]

 [[182 189 192 ... 204 196 190]
  [184 189 192 ... 203 194 184]
  [181 185 190 ... 201 193 183]
  ...
  [104  26  23 ... 158 154 144]
  [123  14  66 ... 158 152 144]
  [129  15 114 ... 150 145 1

In [7]:
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import LabelEncoder

def split_train_test(images, labels):
    ordinal_encoder = LabelEncoder()
    labels_encoded = ordinal_encoder.fit_transform(labels)

    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(sss.split(images, labels_encoded))
    X_train, X_test = images[train_idx], images[test_idx]
    y_train, y_test = labels_encoded[train_idx], labels_encoded[test_idx]
    # X_train_flat = X_train.reshape(len(X_train), -1)
    # X_test_flat  = X_test.reshape(len(X_test), -1)

    return X_train, y_train, X_test, y_test

ordinal_encoder = LabelEncoder()
labels_encoded = ordinal_encoder.fit_transform(labels)

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(sss.split(images, labels_encoded))

X_train, X_test = images[train_idx], images[test_idx]
y_train, y_test = labels_encoded[train_idx], labels_encoded[test_idx]

print("train:", X_train.shape, y_train.shape)
print("test: ", X_test.shape, y_test.shape)


train: (24447, 64, 64) (24447,)
test:  (6112, 64, 64) (6112,)


In [8]:
X_train_flat = X_train.reshape(len(X_train), -1)
X_test_flat  = X_test.reshape(len(X_test), -1)

In [9]:
print(X_train_flat.shape)
print(X_test_flat.shape)

(24447, 4096)
(6112, 4096)


In [10]:
def display_scores(scores):
    print("Scores:", scores)
    print("Mean:", scores.mean())
    print("Standard deviation:", scores.std())

In [11]:
from sklearn.metrics import precision_score, recall_score, f1_score

def display_clf_scores(labels, predictions):
    precision = precision_score(labels, predictions)
    recall = recall_score(labels, predictions)
    f1 = f1_score(labels, predictions)

    print("Precision:", precision)
    print("Recall:", recall)
    print("F1:", f1)


In [ ]:
from typing import Dict, Any
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import SGDClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier


def get_default_classifiers(random_state: int = 42) -> Dict[str, Any]:
    """
    Return a dictionary of main classification models (some with scaling).
    """
    return {
        "gaussian_nb": GaussianNB(),
        "log_reg": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000, random_state=random_state))
        ]),
        "linear_svc": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SGDClassifier(loss="hinge", random_state=random_state))
        ]),
        "svc_rbf": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(kernel="rbf", probability=True, random_state=random_state))
        ]),
        "knn": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier())
        ]),
        "decision_tree": DecisionTreeClassifier(random_state=random_state),
        "random_forest": RandomForestClassifier(random_state=random_state),
        "gradient_boosting": GradientBoostingClassifier(random_state=random_state,
                                                        n_estimators=150,
                                                        learning_rate=0.1,
                                                        max_depth=3,
                                                        subsample=0.7
                                                        ),
        "hist_gradient_boosting": HistGradientBoostingClassifier(random_state=random_state),
        "mlp": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", MLPClassifier(max_iter=500, random_state=random_state))
        ])
    }


def evaluate_classifiers(X, y, scoring: str = "f1_macro", cv: int = 3,
                         n_jobs: int = -1, random_state: int = 42,
                         verbose: int = 1
) -> pd.DataFrame:
    """
    Evaluate many classification models with default hyperparameters.
    Returns a sorted dataframe of cross-validation results.
    """
    classifiers = get_default_classifiers(random_state)
    cv_obj = StratifiedKFold(n_splits=cv, shuffle=True, random_state=random_state)

    results = []

    for name, clf in tqdm(classifiers.items(), total=len(classifiers.items())):
        if verbose:
            print(f"\nEvaluating model: {name}")

        t0 = time.perf_counter()

        scores = cross_val_score(
            clf, X, y,
            scoring=scoring,
            cv=cv_obj,
            n_jobs=n_jobs
        )

        t1 = time.perf_counter()

        results.append({
            "model": name,
            "mean_score": np.mean(scores),
            "std_score": np.std(scores),
            "time_elapsed": t1 - t0
        })

    return pd.DataFrame(results).sort_values("mean_score", ascending=False)


In [21]:
results = evaluate_classifiers(X_train_flat, y_train, scoring="f1_macro", cv=5)
print(results)



Evaluating model: gaussian_nb



Evaluating model: log_reg


/Users/lucasnseyep/code/LucasNseyep/belle-weder/.venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/lucasnseyep/code/LucasNseyep/belle-weder/.venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/lucasnseyep/code/LucasNseyep/belle-weder/.venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/lucasnseyep/code/LucasNseyep/belle-weder/.venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWarning: divide by zero encountered in matmul
  grad[:, :n_features] = grad_pointwise.T @ X + l2_reg_strength 


Evaluating model: linear_svc


/Users/lucasnseyep/code/LucasNseyep/belle-weder/.venv/lib/python3.13/site-packages/sklearn/linear_model/_stochastic_gradient.py:726: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/Users/lucasnseyep/code/LucasNseyep/belle-weder/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/lucasnseyep/code/LucasNseyep/belle-weder/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/lucasnseyep/code/LucasNseyep/belle-weder/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/lucasnseyep/code/LucasNseyep/belle-weder/.venv/lib/python3.13/site-packages/sklearn/linear_model/_stochastic_gradient.py:726: ConvergenceWarning: Maximum number of iteration reached bef

...


Evaluating model: gradient_boosting



Evaluating model: hist_gradient_boosting



Evaluating model: mlp


/Users/lucasnseyep/code/LucasNseyep/belle-weder/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/lucasnseyep/code/LucasNseyep/belle-weder/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/lucasnseyep/code/LucasNseyep/belle-weder/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/lucasnseyep/code/LucasNseyep/belle-weder/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/lucasnseyep/code/LucasNseyep/belle-weder/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/lucasnseyep/code/LucasNseyep/belle-weder/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: 

                    model  mean_score  std_score  time_elapsed
6           random_forest    0.351824   0.007236     54.431549
8  hist_gradient_boosting    0.341735   0.008195    331.689489
7       gradient_boosting    0.309878   0.007445  15190.183959
3                 svc_rbf    0.285913   0.007488   2731.531461
9                     mlp    0.254998   0.008581     69.287898
4                     knn    0.244690   0.007473      8.823068
5           decision_tree    0.240315   0.004517     53.000187
1                 log_reg    0.196650   0.007718     85.559348
2              linear_svc    0.149706   0.007353    381.816555
0             gaussian_nb    0.127946   0.003663      2.014178
